In [18]:
import numpy as np
import torch
import matplotlib.pyplot as plt

from torch.utils import data
from torchvision import datasets,transforms,utils

In [3]:
transform=transforms.Compose([transforms.ToTensor()])

In [5]:
tr_ds=datasets.FashionMNIST(root='./data',train=True,download=True,transform=transform)

In [ ]:
tt_ds=datasets.FashionMNIST(root='./data',train=False,download=True,transform=transform)

In [7]:
tr_ds

Dataset FashionMNIST
    Number of datapoints: 60000
    Root location: ./data
    Split: Train
    StandardTransform
Transform: Compose(
               ToTensor()
           )

In [8]:
tr_ds_loader=data.DataLoader(dataset=tr_ds,batch_size=16)
tt_ds_loader=data.DataLoader(dataset=tt_ds,batch_size=16)

In [9]:
tr_ds_loader,tt_ds_loader

(<torch.utils.data.dataloader.DataLoader at 0x20b6772b1d0>,
 <torch.utils.data.dataloader.DataLoader at 0x20b6527af90>)

In [12]:
ds=iter(tr_ds_loader)
img,label=next(ds)

In [13]:
img.shape,label.shape

(torch.Size([16, 1, 28, 28]), torch.Size([16]))

In [14]:
label

tensor([9, 0, 0, 3, 0, 2, 7, 2, 5, 5, 0, 9, 5, 5, 7, 9])

In [16]:
ans={
0 : '티셔츠/탑 (T-shirt/top)',
1 : '트라우저 (Trouser)',
2 : '풀오버 (Pullover)',
3 : '드레스 (Dress)',
4 : '코트 (Coat)',
5 : '샌들 (Sandal)',
6 : '셔츠 (Shirt)',
7 : '스니커즈 (Sneaker)',
8 : '가방 (Bag)',
9 : '앵클 부츠 (Ankle boot)'
}

In [18]:
idx=label[0].item()

In [19]:
ans[idx]

'앵클 부츠 (Ankle boot)'

In [19]:
from torchvision import transforms,datasets
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

In [20]:
USE_CUDA=torch.cuda.is_available()
DEVICE=torch.device('cuda' if USE_CUDA else 'cpu')

In [21]:
# 작업 준비
BATCH_SIZE=32

In [ ]:
# 데이터 준비
transform=transforms.Compose([transforms.ToTensor()])
tr_ds=datasets.FashionMNIST(root='./data',
                            train=True,
                            download=False,
                            transform=transform)
tt_ds=datasets.FashionMNIST(root='./data',
                            train=False,
                            download=False,
                            transform=transform)

tr_ds_loader=torch.utils.data.DataLoader(dataset=tr_ds,
                                         batch_size=BATCH_SIZE,
                                         shuffle=True)
tt_ds_loader=torch.utils.data.DataLoader(dataset=tt_ds,
                                         batch_size=BATCH_SIZE,
                                         shuffle=True)

In [35]:
# 모델 설계 -> __init__ 설계 후 forward 설계 필수!
class mod(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1=nn.Linear(784,256)
        self.fc2=nn.Linear(256,128)
        self.fc3=nn.Linear(128,10)

    def forward(self,x):
        x=x.view(-1,784)
        x=F.relu(self.fc1(x))
        x=F.relu(self.fc2(x))
        x=self.fc3(x)
        return x

In [ ]:
# 모델설계와 딥러닝 층 쌓는 것과 차이가 뭔지?

In [33]:
DEVICE

device(type='cpu')

In [36]:
m=mod().to(DEVICE)
opt=optim.SGD(m.parameters(),lr=0.01)

In [38]:
for i in tt_ds_loader:
    print(f'x : {i[0].shape}')
    print(f'y : {i[1].shape}')
    break

x : torch.Size([32, 1, 28, 28])
y : torch.Size([32])


In [39]:
def train(m,tr_ds_loader,opt):
    m.train()
    for i,(data,target) in enumerate(tr_ds_loader):
        data,target=data.to(DEVICE),target.to(DEVICE)
        opt.zero_grad()
        output=m(data)
        loss=F.cross_entropy(output,target)
        loss.backward()
        opt.step()   

In [41]:
def evaluate(m,tt_ds_loader):
    m.eval()
    test_loss=0
    correct=0
    with torch.no_grad():
        for data,target in tt_ds_loader:
            output=m(data)

            test_loss+=F.cross_entropy(output,target,reduction='sum').item()
            pred=output.max(1,keepdim=True)[1]
            correct+=pred.eq(target.view_as(pred)).sum().item()
    
    test_loss/=len(tt_ds_loader.dataset)
    test_accuracy=correct/len(tt_ds_loader.dataset)*100.
    return test_loss,test_accuracy

In [43]:
EPOCHS=10
for i in range(1,EPOCHS+1):
    train(m,tr_ds_loader,opt)
    test_loss,test_acc=evaluate(m,tt_ds_loader)
    print(f'epochs : {i} / test_loss : {test_loss} / test_accuracy : {test_acc}')

epochs : 1 / test_loss : 0.40138770322799683 / test_accuracy : 85.65
epochs : 2 / test_loss : 0.4028101739168167 / test_accuracy : 85.78
epochs : 3 / test_loss : 0.3986575170993805 / test_accuracy : 85.84
epochs : 4 / test_loss : 0.38490985999107363 / test_accuracy : 86.22
epochs : 5 / test_loss : 0.37419920697212217 / test_accuracy : 86.75
epochs : 6 / test_loss : 0.37044427754879 / test_accuracy : 86.75
epochs : 7 / test_loss : 0.3633585850715637 / test_accuracy : 87.03999999999999
epochs : 8 / test_loss : 0.3681407132148743 / test_accuracy : 86.65
epochs : 9 / test_loss : 0.353277548623085 / test_accuracy : 87.38
epochs : 10 / test_loss : 0.3526771957397461 / test_accuracy : 87.03999999999999


In [ ]:
# 데이터 로드 후 모델층 완전연결층 구조로 4층 쌓은 DNN 구조 설계 / 학습 후 모델 예측 및 검증

In [53]:
from torchvision import transforms,datasets
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

USE_CUDA=torch.cuda.is_available()
DEVICE=torch.device('cuda' if USE_CUDA else 'cpu')

# 작업 준비
BATCH_SIZE=32

# 데이터 준비
transform=transforms.Compose([transforms.ToTensor()])
tr=datasets.FashionMNIST(root='./data',
                            train=True,
                            download=False,
                            transform=transform)
tt=datasets.FashionMNIST(root='./data',
                            train=False,
                            download=False,
                            transform=transform)

tr_ld=torch.utils.data.DataLoader(dataset=tr,
                                         batch_size=BATCH_SIZE,
                                         shuffle=True)
tt_ld=torch.utils.data.DataLoader(dataset=tt,
                                         batch_size=BATCH_SIZE,
                                         shuffle=True)

# 모델 설계 -> __init__ 설계 후 forward 설계 필수!
class model(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1=nn.Linear(784,512)
        self.fc2=nn.Linear(512,256)
        self.fc3=nn.Linear(256,128)
        self.fc4=nn.Linear(128,10)

    def forward(self,x):
        x=x.view(-1,784)
        x=F.relu(self.fc1(x))
        x=F.relu(self.fc2(x))
        x=F.relu(self.fc3(x))
        x=self.fc4(x)
        return x
    
krap=model().to(DEVICE)
opt=optim.SGD(krap.parameters(),lr=0.01)

def train(krap,tr_ld,opt):
    krap.train()
    for i,(data,target) in enumerate(tr_ld):
        data,target=data.to(DEVICE),target.to(DEVICE)
        opt.zero_grad()
        output=krap(data)
        loss=F.cross_entropy(output,target)
        loss.backward()
        opt.step()

def evaluate(krap,tt_ld):
    krap.eval()
    loss=0
    correct=0
    with torch.no_grad():
        for data,target in tt_ld:
            output=krap(data)

            loss+=F.cross_entropy(output,target,reduction='sum').item()
            pred=output.max(1,keepdim=True)[1]
            correct+=pred.eq(target.view_as(pred)).sum().item()
    
    loss/=len(tt_ld.dataset)
    accuracy=correct/len(tt_ld.dataset)*100.
    return loss,accuracy

EPOCHS=10
for i in range(1,EPOCHS+1):
    train(krap,tr_ld,opt)
    loss,acc=evaluate(krap,tt_ld)
    print(f'epochs : {i} / loss : {loss} / accuracy : {acc}')

epochs : 1 / loss : 0.7473678864479065 / accuracy : 70.87
epochs : 2 / loss : 0.5963562305450439 / accuracy : 78.94
epochs : 3 / loss : 0.5279731870651245 / accuracy : 81.35
epochs : 4 / loss : 0.5015450296401978 / accuracy : 82.15
epochs : 5 / loss : 0.5004935251712799 / accuracy : 81.39
epochs : 6 / loss : 0.43930242807865144 / accuracy : 84.38
epochs : 7 / loss : 0.41616237444877624 / accuracy : 84.99
epochs : 8 / loss : 0.4368326442718506 / accuracy : 83.89999999999999
epochs : 9 / loss : 0.4035673653364181 / accuracy : 85.41
epochs : 10 / loss : 0.38691010251045227 / accuracy : 86.07000000000001


In [22]:
#데이터 준비
transform=transforms.Compose([
    transforms.ToTensor()
])#데이터 사용 방식 내용 결정
tr_ds=datasets.FashionMNIST(
    root='./data/',
    train=True,
    download=False,
    transform=transform
)
tt_ds=datasets.FashionMNIST(
    root='./data/',
    train=False,
    download=False,
    transform=transform
)
tr_ds_loader=torch.utils.data.DataLoader(
    dataset=tr_ds,
    batch_size=BATCH_SIZE,
    shuffle=True
)
tt_ds_loader=torch.utils.data.DataLoader(
    dataset=tt_ds,
    batch_size=BATCH_SIZE,
    shuffle=True
)

#모델 설계
class 모델(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1=nn.Linear(784,256)
        self.fc2=nn.Linear(256,128)
        self.fc3=nn.Linear(128,10)
    def forward(self,x):
        x=x.view(-1,784)
        x=F.relu(self.fc1(x))
        x=F.relu(self.fc2(x))
        x=self.fc3(x)
        return x
#모델 생성
m=모델().to(DEVICE)
#최적화 함수 정의
opt=optim.SGD(m.parameters(),lr=0.01) 
#컴파일
def train(m,tr_ds_loader,opt):
    m.train()
    for i,(data,target) in enumerate(tr_ds_loader):
        data,target=data.to(DEVICE),target.to(DEVICE)
        opt.zero_grad()
        output=m(data)
        loss=F.cross_entropy(output,target)
        loss.backward()
        opt.step()
        
#오차 계산 시각화
def evaluate(m,tt_ds_loader):
    m.eval()
    test_loss=0
    correct=0
    with torch.no_grad():
        for data,target in tt_ds_loader:
            data,target=data.to(DEVICE),target.to(DEVICE)
            output=m(data)

            test_loss += F.cross_entropy(output,target,reduction='sum').item()
            pred=output.max(1,keepdim=True)[1]
            correct += pred.eq(target.view_as(pred)).sum().item()
    test_loss/=len(tt_ds_loader.dataset)
    test_accuracy= correct/len(tt_ds_loader.dataset)*100.
    return test_loss,test_accuracy
    
#학습 진행    
EPOCHS=10
for i in range(1,EPOCHS+1):
    train(m,tr_ds_loader,opt)
    test_loss,test_acc=evaluate(m,tt_ds_loader)

    print(f'{i}epoch test_loss:{test_loss},test_accuracy:{test_acc}')

#예측및 검증
py=m(data)
F.log_softmax(py)

1epoch test_loss:0.6648123670578003,test_accuracy:76.85
2epoch test_loss:0.5372124063014985,test_accuracy:81.25
3epoch test_loss:0.5077388112068176,test_accuracy:81.69
4epoch test_loss:0.4811997107744217,test_accuracy:82.69
5epoch test_loss:0.4565883688688278,test_accuracy:83.66
6epoch test_loss:0.4513509657859802,test_accuracy:83.98
7epoch test_loss:0.4489441803455353,test_accuracy:84.13000000000001
8epoch test_loss:0.41833426489830017,test_accuracy:85.22
9epoch test_loss:0.42044834156036376,test_accuracy:84.89999999999999
10epoch test_loss:0.422726727938652,test_accuracy:84.74000000000001


AttributeError: module 'torch.utils.data' has no attribute 'view'

In [ ]:
# 작업 준비(이미지 처리)
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torchvision import transforms, datasets

import numpy as np
import matplotlib.pyplot as plt

In [ ]:
BATCH_SIZE=60000
tr_ds_loader=torch.utils.data.DataLoader(
    dataset=tr_ds,
    batch_size=BATCH_SIZE,
    shuffle=False
)
img,_=next(iter(tr_ds_loader))
img.shape
img.mean(),img.std()

In [ ]:
img.mean(),img.std()

In [ ]:
# 데이터 수정(노이즈 삽입)
# 데이터 준비
transform=transforms.Compose([
    transforms.RandomHorizontalFlip(), # 데이터 증강(노이즈삽입)
    transforms.ToTensor(), # 입력 데이터 정리
    transforms.Normalize((0.2860,),(0.3530,))
])

# 데이터 사용 방식 내용 결정
tr_ds_loader=torch.utils.data.DataLoader(
    datasets.FashionMNIST('./data',
        train=True,
        download=False,
        transform=transform
    ),
    batch_size=BATCH_SIZE,
    shuffle=True
)

tt_ds_loader=torch.utils.data.DataLoader(
    datasets.FashionMNIST('./data',
        train=True,
        download=False,
        transform=transform
    ),
    batch_size=BATCH_SIZE,
    shuffle=True
)

In [60]:
x=0
def f(x):
    return x+1

for x in [1,2,3]:
    f(x)
    print(x)

1
2
3


In [ ]:
class DNN_model(nn.Module):
    def __init__(self,input_n,hidden_ns,output_n,dropout_p=0.2):
        super().__init__()
        self.fc_in=nn.Linear(input_n,hidden_ns[0])
        self.fc_h_l=[nn.Linear(hidden_ns[i],hidden_ns[i+1]) for i in range(len(hidden_ns)-1)]
        self.fc_out=nn.Linear(hidden_ns[-1],output_n)

        self.dropout_p=dropout_p
        self.input_n=input_n
        self.hidden_ns=hidden_ns
        self.output_n=output_n
    
    def forward(self,x):
        x=x.view(-1,self.input_n) # 벡터화
        x=F.relu(self.fc_in(x)) # 입력계층 연산
        x=F.dropout(x,training=self.training,p=self.dropout_p)

        for i in range(len(self.hidden_ns)-1): # 은닉계층 연산
            x=F.relu(self.fc_h_l[i](x))
            x=F.dropout(x,training=self.training,p=self.dropout_p)
        out=self.fc_out(x) # 출력계층 연산
        return out

In [15]:
m=DNN_model(784,[256,128,68],10).to(DEVICE)
opt=optim.SGD(m.parameters(),lr=0.01)

In [ ]:
def train(m,tr_ds_loader,opt):
    m.train()
    for i,(x,y) in enumerate(tr_ds_loader):
        data,target=x.to(DEVICE),y.to(DEVICE)
        opt.zero_grad()
        py=m(data)
        loss=F.cross_entropy(py,target)
        loss.backward()
        opt.step()

def evaluate(m,tt_ds_loader):
    m.eval()
    test_loss=0
    correct=0
    with torch.no_grad():
        for data,target in tt_ds_loader:
            data,target=data.to(DEVICE),target.to(DEVICE)
            py=m(data)
            test_loss+=F.cross_entropy(py,target,reduction='sum').item()
            pred=py.max(1,keepdim=True)[1]
            correct+=pred.eq(target.view_as(pred)).sum().item()
    test_loss/=len(tt_ds_loader.dataset)
    test_accuracy=correct/len(tt_ds_loader.dataset)*100.
    return test_loss,test_accuracy   

In [ ]:
EPOCHS=10
for i in range(1,EPOCHS+1):
    train(m,tr_ds_loader,opt)
    test_loss,test_acc=evaluate(m,tt_ds_loader)

    print(f'{i}epoch test_loss:{test_loss},test_accuracy:{test_acc}')

epochs : 1 / test_loss : 1.102783206017812 / test_accuracy : 65.755
epochs : 2 / test_loss : 0.8032481187661489 / test_accuracy : 72.13666666666667
epochs : 3 / test_loss : 0.6999303212801615 / test_accuracy : 73.82333333333332
epochs : 4 / test_loss : 0.6442141288757324 / test_accuracy : 76.01833333333333
epochs : 5 / test_loss : 0.6073335514704387 / test_accuracy : 77.42666666666666


KeyboardInterrupt: 

In [ ]:
@torch.no_grad()#서식
def evaluate(m,tt_ds_loader):
    m.eval()
    test_loss=0
    correct=0
    
    for data,target in tt_ds_loader:
        data,target=data.to(DEVICE),target.to(DEVICE)
        py=m(data)
        test_loss+=F.cross_entropy(py,target,reduction='sum').item()
        pred=py.max(1,keepdim=True)[1]
        correct += pred.eq(target.view_as(pred)).sum().item()
    test_loss/=len(tt_ds_loader.dataset)
    test_accuracy= correct/len(tt_ds_loader.dataset)*100.
    return test_loss,test_accuracy   